In [5]:
import json
import numpy as np
import pandas as pd
from langdetect import detect, detect_langs
from copy import deepcopy

In [6]:
with open("data/all_results.json", "r", encoding="utf-8") as f:
            survey_full_results = json.load(f)

In [7]:
# 1. Setup the Master Answer Key
# Combining both languages into one set for high-speed, order-independent lookup
attn_chk_pass_answer_DK = [
    'En person, som ikke er fra et politisk parti, men støtter et parti.',
    "En person fra NGO'er, der fremmer valgdeltagelse."
]
attn_chk_pass_answer_EN = [
    'Someone who is not from a political party but supports a party.',
    'Someone from NGOs promoting electoral participation.'
]
valid_answers = set(attn_chk_pass_answer_DK + attn_chk_pass_answer_EN)

In [8]:
# 2. Initialize counters and lists
fail_count = 0
pass_count = 0
no_pass_list = []

# 3. Process the results
for res in survey_full_results:
    # Safely retrieve the list of answers from the nested dictionary
    attn_chk_res = res['surveyData']['votingExperience']['ATTN_CHK']
    if detect(attn_chk_res[0]) == 'en':  # output: 'en' for english; 'da' for danish
        res['surveyData']['language'] = 'EN'
    else: 
        res['surveyData']['language'] = 'DA'
    
    # Calculate overlap using set intersection
    # This finds which of the respondent's answers are in our 'valid_answers' set
    correct_selections = set(attn_chk_res) & valid_answers
    num_correct = len(correct_selections)
    total_selected = len(attn_chk_res)

    # 1. PASS: Exactly 2 correct and ONLY 2 selected.
    if num_correct == 2 and total_selected == 2:
        res['ATTN_CHK_PF'] = 'PASS'
        res['ATTN_CHK_FAIL_REASON'] = None
        pass_count += 1
        
    # 2. HALF_PASS: Picked 1 or 2 correct, but total selections must be less than 4.
    # This keeps people who picked (Correct + Incorrect) or (Correct + Correct + Incorrect).
    elif num_correct >= 1 and total_selected < 4:
        res['ATTN_CHK_PF'] = 'HALF_PASS'
        res['ATTN_CHK_FAIL_REASON'] = None
        no_pass_list.append(attn_chk_res)
        
    # 3. FAIL: Either zero correct OR they "gamed" the system by picking 4+ options.
    else:
        res['ATTN_CHK_PF'] = 'FAIL'
        # Optional: Add a specific reason for your internal tracking
        if total_selected >= 4:
            res['ATTN_CHK_FAIL_REASON'] = 'Over-selection (4+ items)'
        else:
            res['ATTN_CHK_FAIL_REASON'] = 'Zero correct matches'
            
        no_pass_list.append(attn_chk_res)
        fail_count += 1

# 4. Optional: Print summary
print(f"Results processed: {pass_count} Pass, {fail_count} Fail.")

Results processed: 557 Pass, 395 Fail.


In [10]:
column_name_list = []
initial_survey_data = survey_full_results[0]['surveyData']
for col_nam in initial_survey_data.keys():
    if type(initial_survey_data[col_nam]) is dict:
        column_name_list = column_name_list + list(initial_survey_data[col_nam].keys())
    else: column_name_list.append(col_nam)
column_name_list = column_name_list + list(survey_full_results[0].keys())[-2:]

In [11]:
translation_dict = {
    'SCREEN_ELIGIBILITY': {"EN": ['Yes'],
              "DA": ['Ja']},
    'SCREEN_RESIDENCE': {"EN": ['Yes'],
              "DA": ['Ja']},
    'ALCL11': {"EN": ['Municipal tax in Aarhus Municipality must be raised, and the money must be spent on better welfare.', 'It is possible to save money in the public sector without affecting public welfare.',
                     'More tasks in the public sector must be solved by private companies.', 'Aarhus Municipality must make it cheaper to run a business.',
                     'Aarhus Municipality must prioritize that school pupils are mixed according to ethnicity and social background.', 'The politicians must prevent the construction of mosques.',
                     'More parking spaces must be established in Aarhus Municipality.', 'The city council’s temporary stop for the expansion of Aarhus Harbor must be made permanent.',
                     'Car traffic in Aarhus city center must be limited, for example through one-way directions, speed reductions and a zero-emission zone.',
                     'The municipality must continue with the plans for the new football stadium in Kongelunden, even if the costs rise again.'],
              "DA": ['Kommuneskatten i Aarhus Kommune skal hæves, og pengene skal bruges på bedre velfærd.', 'Det er muligt at spare penge i den offentlige sektor uden at påvirke den offentlige velfærd.',
                     'Flere opgaver i den offentlige sektor skal løses af private virksomheder.', 'Aarhus Kommune skal gøre det billigere at drive virksomhed.',
                     'Aarhus Kommune skal prioritere, at skoleelever blandes på tværs af etnicitet og social baggrund.', 'Politikerne skal forhindre opførelsen af moskéer.',
                     'Der skal etableres flere parkeringspladser i Aarhus Kommune.', 'Byrådets midlertidige stop for udvidelsen af Aarhus Havn skal gøres permanent.',
                     'Biltrafikken i Aarhus centrum skal begrænses, for eksempel gennem ensretninger, hastighedsnedsættelser og en nulemissionszone.',
                     'Kommunen skal fortsætte planerne om det nye fodboldstadion i Kongelunden, selv hvis omkostningerne stiger igen.']},
    'VOTE_LOC': {"EN": ['I did not vote.', 'I thought about voting this time, but I didn’t.', 'I usually vote but didn’t this time.', 'I am sure I voted.'],
              "DA": ['Jeg stemte ikke.', 'Jeg overvejede at stemme denne gang, men jeg gjorde det ikke.', 
                     'Jeg plejer at stemme, men gjorde det ikke denne gang.', 'Jeg er sikker på, at jeg stemte.']},
    'VOTE_NAT': {"EN": ['I did not vote.', 'I thought about voting this time, but I didn’t.', 'I usually vote but didn’t this time.', 'I am sure I voted.'],
              "DA": ['Jeg stemte ikke.', 'Jeg overvejede at stemme denne gang, men jeg gjorde det ikke.', 
                     'Jeg plejer at stemme, men gjorde det ikke denne gang.', 'Jeg er sikker på, at jeg stemte.']},
    'ATTN_CHK': {"EN": ['Someone from a political party.', 'Someone who is not from a political party but supports a party.',
                       'Someone from NGOs promoting electoral participation.', 'No one from a political party.'],
              "DA": ['En person fra et politisk parti.', 'En person, som ikke er fra et politisk parti, men støtter et parti.',
                    "En person fra NGO'er, der fremmer valgdeltagelse.", 'Ingen fra et politisk parti.']},
    'GENDER': {"EN": ['Male', 'Female', 'Non-binary', 'Prefer not to say'],
              "DA": ['Mand', 'Kvinde', 'Non-binær', 'Ønsker ikke at svare']},
    'EDUCATION': {"EN": ['Primary and lower secondary education (e.g., Folkeskole or Friskole)', 'Upper secondary education (e.g., STX, HTX, HHX, HF)',
                        'Vocational education and training (EUD or EUX)', 'Short-cycle higher education (1-2 years)', 'Medium-length higher education (3-4 years)',
                        'Long-cycle higher education (5-7 years)', 'PhD or other research degree.', 'Other', 'Prefer not to say'],
              "DA": ['Grundskole eller tilsvarende (f.eks. Folkeskole eller Friskole)', 'Gymnasial uddannelse (f.eks. STX, HTX, HHX, HF)',
                    'Erhvervsuddannelse (f.eks. EUD eller EUX)', 'Kort videregående uddannelse (1-2år)',
                    'Mellemlang videregående uddannelse (3–4 år)', 'Lang videregående uddannelse (5–7 år)',
                    'PhD anden forskeruddannelse', 'Andet', 'Ønsker ikke at svare']},
    'JOB': {"EN": ['In paid work (employee, self-employed, working for your family business) or temporary absent.', 'In education (not paid for by employer), even if on vacation',
                  'Unemployed and actively looking for a job', 'Unemployed, wanting a job but not actively looking for a job', 'Permanently sick or disabled',
                  'Retired', 'In community or military service', 'Doing housework, looking after children or other persons', 'Other', 'Prefer not to say'],
              "DA": ['I lønnet arbejde (ansat, selvstændig eller i familiens virksomhed), også midlertidigt fraværende', 
                     'Under uddannelse (ikke betalt af arbejdsgiver), også hvis du har haft ferie', 'Arbejdsløs og aktivt jobsøgende',	
                     'Arbejdsløs, ønsker job, søger ikke aktivt', 'Varigt syg eller med handicap', 'Pensioneret',
                     'I samfunds- eller militærtjeneste', 'Hjemmegående/passer børn eller andre personer', 'Andet', 'Ønsker ikke at svare']},
    'SRQV1': {"EN": ['It should only be allowed to give one vote to each candidate.', 'It should be allowed to give maximally two or three votes to one.',
              'It should be allowed to give more votes to one candidate, but an additional vote should "cost" more than one vote.', 
              'It should be allowed to give all votes to just one candidate without additional "costs".'],
              "DA": ['Det bør kun være tilladt at give én stemme til hver kandidat.', 'Det bør være tilladt at give højst to eller tre stemmer til én kandidat.',
                     'Det bør være tilladt at give flere stemmer til én kandidat, men en ekstra stemme bør "koste" mere end én stemme.',
                     'Det bør være tilladt at give alle stemmer til én kandidat uden ekstra "omkostninger".']}
}

In [13]:
survey_df_dict = {col_nam: [] for col_nam in column_name_list}

for res in survey_full_results:
    for col_nam in column_name_list:
        if "ATTN_CHK_" in col_nam:
            survey_df_dict[col_nam].append(res[col_nam])
        elif col_nam == 'SRQV1':
            survey_df_dict[col_nam].append(res['srqv1'])
        else:
            if col_nam in res['surveyData'].keys():
                survey_df_dict[col_nam].append(res['surveyData'][col_nam])
            else:
                for key in res['surveyData'].keys():
                    if type(res['surveyData'][key]) is dict and col_nam in list(res['surveyData'][key].keys()):
                        survey_df_dict[col_nam].append(res['surveyData'][key][col_nam])

In [14]:
survey_df_dict_ENG = deepcopy(survey_df_dict)
for trans_col_nam in translation_dict.keys():
    for i, res in enumerate(survey_df_dict_ENG[trans_col_nam]):
        if type(res) is list:
            for j, re in enumerate(res):
                if survey_df_dict_ENG['language'][i] == 'DA':
                    if re in translation_dict[trans_col_nam]['EN']:
                        pass
                    else:
                        eng_re = translation_dict[trans_col_nam]['EN'][translation_dict[trans_col_nam]['DA'].index(re)]
                        res[j] = eng_re
        else:
            if res == 'Ja' or res == 'Yes':
                survey_df_dict_ENG[trans_col_nam][i] = 'Yes'
            else:
                if survey_df_dict_ENG['language'][i] == 'DA':
                    if res in translation_dict[trans_col_nam]['EN']:
                        pass
                    elif res == '':
                        pass
                    else:
                        survey_df_dict_ENG[trans_col_nam][i] = translation_dict[trans_col_nam]['EN'][translation_dict[trans_col_nam]['DA'].index(res)]               

In [15]:
df_survey_response = pd.DataFrame.from_dict(survey_df_dict); df_survey_response

,userId,surveyDuration,AGE,SCREEN_ELIGIBILITY,SCREEN_RESIDENCE,NSEC_B,NSEC_BCRT,NSEC_P,NECN_B,NECN_BCRT,...,SSBS1,SSBS2,SQDR1,GENDER,EDUCATION,JOB,SRQV1,language,ATTN_CHK_PF,ATTN_CHK_FAIL_REASON
0,3a203770e837d765cd5f6f984a4f598c,406.388,57,Ja,Ja,7,6,5,6,6,...,7,5,5,Kvinde,PhD anden forskeruddannelse,"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør være tilladt at give alle stemmer til ...,DA,FAIL,Zero correct matches
1,3a203770e80d24440b6379c848d401f9,463.966,77,Ja,Ja,7,8,8,10,10,...,10,10,0,Mand,Erhvervsuddannelse (f.eks. EUD eller EUX),[Pensioneret],Det bør kun være tilladt at give én stemme til...,DA,FAIL,Zero correct matches
2,3a203770e7e805db58c55016db059137,859.743,41,Ja,Ja,6,10,2,4,10,...,5,6,6,Mand,Erhvervsuddannelse (f.eks. EUD eller EUX),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør kun være tilladt at give én stemme til...,DA,FAIL,Zero correct matches
3,3a203770e848ec257fce0316dd4cfd65,276.418,48,Ja,Ja,7,9,9,6,9,...,10,10,0,Kvinde,Erhvervsuddannelse (f.eks. EUD eller EUX),"[I lønnet arbejde (ansat, selvstændig eller i ...",It should be allowed to give all votes to just...,DA,PASS,None
4,3a203770e7efbe8b69127c396a4165f4,341.944,44,Ja,Ja,10,10,0,9,10,...,10,9,1,Mand,Mellemlang videregående uddannelse (3–4 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør kun være tilladt at give én stemme til...,DA,PASS,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,3a203c634ef0953007536acb9b60f568,624.978,27,Ja,Ja,8,6,5,6,5,...,6,5,5,Kvinde,Mellemlang videregående uddannelse (3–4 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør være tilladt at give alle stemmer til ...,DA,FAIL,Zero correct matches
1015,3a2054ba33aaf5d31419ee397ce605f6,1805.128,34,Ja,Ja,6,5,4,10,10,...,7,7,3,Kvinde,Mellemlang videregående uddannelse (3–4 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",Det bør være tilladt at give flere stemmer til...,DA,PASS,None
1016,3a20644a9722a8e51052ba2f2ff0aeff,396.447,34,Ja,Ja,6,5,3,3,5,...,7,4,5,Mand,Lang videregående uddannelse (5–7 år),"[I lønnet arbejde (ansat, selvstændig eller i ...",It should only be allowed to give one vote to ...,DA,PASS,None
1017,3a20644a97f8894830a72b2fd9f4c03e,707.764,23,Ja,Ja,4,6,6,3,6,...,8,8,8,Mand,"Gymnasial uddannelse (f.eks. STX, HTX, HHX, HF)",[Ønsker ikke at svare],Det bør være tilladt at give højst to eller tr...,DA,PASS,None


In [16]:
df_survey_response_ENG = pd.DataFrame.from_dict(survey_df_dict_ENG); df_survey_response_ENG

,userId,surveyDuration,AGE,SCREEN_ELIGIBILITY,SCREEN_RESIDENCE,NSEC_B,NSEC_BCRT,NSEC_P,NECN_B,NECN_BCRT,...,SSBS1,SSBS2,SQDR1,GENDER,EDUCATION,JOB,SRQV1,language,ATTN_CHK_PF,ATTN_CHK_FAIL_REASON
0,3a203770e837d765cd5f6f984a4f598c,406.388,57,Yes,Yes,7,6,5,6,6,...,7,5,5,Female,PhD or other research degree.,"[In paid work (employee, self-employed, workin...",It should be allowed to give all votes to just...,DA,FAIL,Zero correct matches
1,3a203770e80d24440b6379c848d401f9,463.966,77,Yes,Yes,7,8,8,10,10,...,10,10,0,Male,Vocational education and training (EUD or EUX),[Retired],It should only be allowed to give one vote to ...,DA,FAIL,Zero correct matches
2,3a203770e7e805db58c55016db059137,859.743,41,Yes,Yes,6,10,2,4,10,...,5,6,6,Male,Vocational education and training (EUD or EUX),"[In paid work (employee, self-employed, workin...",It should only be allowed to give one vote to ...,DA,FAIL,Zero correct matches
3,3a203770e848ec257fce0316dd4cfd65,276.418,48,Yes,Yes,7,9,9,6,9,...,10,10,0,Female,Vocational education and training (EUD or EUX),"[In paid work (employee, self-employed, workin...",It should be allowed to give all votes to just...,DA,PASS,None
4,3a203770e7efbe8b69127c396a4165f4,341.944,44,Yes,Yes,10,10,0,9,10,...,10,9,1,Male,Medium-length higher education (3-4 years),"[In paid work (employee, self-employed, workin...",It should only be allowed to give one vote to ...,DA,PASS,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,3a203c634ef0953007536acb9b60f568,624.978,27,Yes,Yes,8,6,5,6,5,...,6,5,5,Female,Medium-length higher education (3-4 years),"[In paid work (employee, self-employed, workin...",It should be allowed to give all votes to just...,DA,FAIL,Zero correct matches
1015,3a2054ba33aaf5d31419ee397ce605f6,1805.128,34,Yes,Yes,6,5,4,10,10,...,7,7,3,Female,Medium-length higher education (3-4 years),"[In paid work (employee, self-employed, workin...",It should be allowed to give more votes to one...,DA,PASS,None
1016,3a20644a9722a8e51052ba2f2ff0aeff,396.447,34,Yes,Yes,6,5,3,3,5,...,7,4,5,Male,Long-cycle higher education (5-7 years),"[In paid work (employee, self-employed, workin...",It should only be allowed to give one vote to ...,DA,PASS,None
1017,3a20644a97f8894830a72b2fd9f4c03e,707.764,23,Yes,Yes,4,6,6,3,6,...,8,8,8,Male,"Upper secondary education (e.g., STX, HTX, HHX...",[Prefer not to say],It should be allowed to give maximally two or ...,DA,PASS,None


In [17]:
df_survey_response.to_csv('survey_responses_original.csv', encoding = 'utf-8')
df_survey_response_ENG.to_csv('survey_responses_ENG_TRANS.csv', encoding = 'utf-8')